# Hubmicroo Voice Assistant — Kaggle Test Environment

> **Testing only** — sessions expire after 12 h. For production use `docker compose up`.

**Before running:**
- Settings → Accelerator → **GPU T4 x2**
- Settings → Internet → **On**

**No Qdrant server needed** — uses qdrant-client embedded (local file) mode.

In [ ]:
# ── Step 1: System dependencies ───────────────────────────────────────────
import subprocess, sys

def run(cmd, check=True):
    """Run a shell command, print last 400 chars of stdout, raise on error if check=True."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout.strip():
        print(r.stdout[-400:])
    if r.returncode != 0:
        print('STDERR:', r.stderr[-300:])
        if check:
            raise RuntimeError(f'Command failed (exit {r.returncode}): {cmd[:80]}')
    return r

print('Installing system deps...')
run('apt-get update -qq && apt-get install -y --no-install-recommends ffmpeg libgomp1 curl', check=False)
print('✅ System deps done')

In [ ]:
# ── Step 2: Python packages ────────────────────────────────────────────────
print('Installing Python packages (torch + ML stack)...')
run('pip install -q torch --index-url https://download.pytorch.org/whl/cu121', check=False)
run('pip install -q fastapi "uvicorn[standard]" pydantic pydantic-settings python-multipart')
run('pip install -q "python-jose[cryptography]" "passlib[bcrypt]" bcrypt')
run('pip install -q qdrant-client rank-bm25 httpx')
run('pip install -q FlagEmbedding transformers sentence-transformers')
run('pip install -q faster-whisper')
run('pip install -q pyngrok requests')

# Confirm qdrant embedded mode works (no server needed)
from qdrant_client import QdrantClient, models as qm
tc = QdrantClient(path='/tmp/_qdrant_test')
tc.create_collection('_t', vectors_config=qm.VectorParams(size=4, distance=qm.Distance.COSINE))
del tc
print('✅ All packages OK | Qdrant embedded mode confirmed')

In [ ]:
# ── Step 3: Install Ollama (direct binary — no install script, no systemd) ──
#
# install.sh requires systemd (absent on Kaggle) → exits non-zero before
# writing the binary. Ollama ships a single flat binary (~500 MB); curl -fsSL
# with no extra flags follows GitHub's multi-hop CDN redirect correctly.
import subprocess, time, os, pathlib, urllib.request

OLLAMA_BIN = '/usr/local/bin/ollama'

if not pathlib.Path(OLLAMA_BIN).exists():
    print('Downloading Ollama binary (~500 MB, please wait)...')
    r = subprocess.run(
        'curl -fsSL -o /usr/local/bin/ollama '
        'https://github.com/ollama/ollama/releases/latest/download/ollama-linux-amd64',
        shell=True, capture_output=True, text=True
    )
    if r.returncode != 0:
        raise RuntimeError(f'curl failed (exit {r.returncode}):\n{r.stderr[-300:]}')
    os.chmod(OLLAMA_BIN, 0o755)
    size_mb = pathlib.Path(OLLAMA_BIN).stat().st_size // 1_000_000
    print(f'✅ Ollama binary downloaded ({size_mb} MB)')
else:
    size_mb = pathlib.Path(OLLAMA_BIN).stat().st_size // 1_000_000
    print(f'✅ Ollama already present ({size_mb} MB)')

# Always use absolute path + explicit PATH so child processes can find it too
env = os.environ.copy()
env['PATH'] = '/usr/local/bin:' + env.get('PATH', '/usr/bin:/bin')
env['OLLAMA_KEEP_ALIVE'] = '5m'
env['HOME'] = '/root'

print('Starting Ollama server...')
ollama_proc = subprocess.Popen(
    [OLLAMA_BIN, 'serve'], env=env,
    stdout=open('/tmp/ollama.log', 'w'), stderr=subprocess.STDOUT
)

print('Waiting for Ollama API...')
for i in range(25):
    time.sleep(3)
    try:
        urllib.request.urlopen('http://localhost:11434/api/tags', timeout=3)
        print(f'✅ Ollama server ready (took ~{(i+1)*3}s)')
        break
    except:
        if i % 4 == 0:
            print(f'  [{(i+1)*3}s] still starting...')
else:
    with open('/tmp/ollama.log') as f:
        print('Ollama log:', f.read()[-600:])
    raise RuntimeError('Ollama server did not start — check log above')

MODEL = 'qwen3:4b'
print(f'Pulling {MODEL}  (~3–8 min first run, instant if cached)...')
r = subprocess.run([OLLAMA_BIN, 'pull', MODEL], env=env,
                   capture_output=True, text=True, timeout=600)
if r.stdout:
    print(r.stdout[-200:])
if r.returncode != 0:
    raise RuntimeError(f'ollama pull failed:\n{r.stderr[-300:]}')
print('✅ Model ready')

In [ ]:
# ── Step 4: Clone repo ────────────────────────────────────────────────────
import sys, os, pathlib, subprocess

REPO_ROOT = pathlib.Path('/kaggle/working/Hubmicroo_VoiceAssistant')

if not REPO_ROOT.exists():
    print('Cloning repo...')
    r = subprocess.run(
        'git clone https://github.com/ai-with-abdullah/Hubmicroo-AI-Voice-Assistant.git '
        + str(REPO_ROOT),
        shell=True, capture_output=True, text=True
    )
    if r.returncode != 0:
        raise RuntimeError(f'Clone failed:\n{r.stderr}')
    print('✅ Repo cloned')
else:
    print('Pulling latest...')
    subprocess.run(f'git -C {REPO_ROOT} pull --ff-only', shell=True)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

DATA_DIR     = str(REPO_ROOT / 'backend' / 'data')
FRONTEND_DIR = str(REPO_ROOT / 'frontend')
QDRANT_DB    = '/tmp/qdrant_db'

env_text = f"""LLM_MODEL=qwen3:4b
OLLAMA_BASE_URL=http://localhost:11434
QDRANT_URL=http://localhost:6333
QDRANT_LOCAL_PATH={QDRANT_DB}
QDRANT_COLLECTION=hubmicroo
EMBED_MODEL=BAAI/bge-m3
EMBED_DEVICE=cuda
WHISPER_DEVICE=cuda
WHISPER_MODEL=base
TTS_ENABLED=false
DATA_DIR={DATA_DIR}
PRODUCTS_FILE={DATA_DIR}/products.json
PAGES_FILE={DATA_DIR}/pages.json
FRONTEND_DIR={FRONTEND_DIR}
ADMIN_USERNAME=admin
ADMIN_PASSWORD=kaggle-test-pass
SECRET_KEY=kaggle-testing-secret-key-32chars
CORS_ORIGINS=*
"""
(REPO_ROOT / '.env').write_text(env_text)
print('✅ .env written')
print('  DATA_DIR      :', DATA_DIR, '→ exists:', pathlib.Path(DATA_DIR).exists())
print('  products.json :', (pathlib.Path(DATA_DIR) / 'products.json').exists())
print('  Qdrant (embed):', QDRANT_DB)

In [ ]:
# ── Step 5: Seed the vector index ────────────────────────────────────────
import sys
# Clear cached backend modules so .env is re-read
for key in [k for k in sys.modules if k.startswith('backend')]:
    del sys.modules[key]

from backend.app.config import get_settings
from backend.app import store
from backend.app.indexer import rebuild_all

s = get_settings()
print('Settings:')
print('  QDRANT_LOCAL_PATH :', s.QDRANT_LOCAL_PATH)
print('  EMBED_DEVICE      :', s.EMBED_DEVICE)
print('  DATA_DIR          :', s.DATA_DIR)

store.ensure_collection()
print('Indexing (BGE-M3 download ~1.5 GB on first run)...')
count = rebuild_all()
stats = store.collection_stats()
print(f'✅ Indexed {count} items | Qdrant points: {stats["points_count"]}')

In [ ]:
# ── Step 6: Start FastAPI backend ─────────────────────────────────────────
import subprocess, time, urllib.request, json, os

subprocess.run('fuser -k 8000/tcp 2>/dev/null || true', shell=True)
time.sleep(1)

env = os.environ.copy()
env['PYTHONPATH'] = str(REPO_ROOT)
env['PATH'] = '/usr/local/bin:' + env.get('PATH', '/usr/bin:/bin')

server = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'backend.app.main:app',
     '--host', '0.0.0.0', '--port', '8000', '--log-level', 'info'],
    cwd=str(REPO_ROOT), env=env,
    stdout=open('/tmp/fastapi.log', 'w'), stderr=subprocess.STDOUT
)

print('Waiting for backend...')
for i in range(40):
    time.sleep(3)
    try:
        with urllib.request.urlopen('http://localhost:8000/api/health', timeout=5) as r:
            h = json.loads(r.read())
        print(f'✅ Backend ready: {h}')
        break
    except Exception as e:
        if i % 5 == 0:
            print(f'  [{(i+1)*3}s] still starting...')
else:
    print('❌ Backend not responding. Last log lines:')
    with open('/tmp/fastapi.log') as f:
        print(''.join(f.readlines()[-30:]))

In [ ]:
# ── Step 7: Expose via ngrok (optional public URL) ────────────────────────
# Get a free token at: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = ''  # <── paste your token here

from pyngrok import ngrok, conf

if NGROK_TOKEN:
    conf.get_default().auth_token = NGROK_TOKEN
    tunnel = ngrok.connect(8000)
    public_url = tunnel.public_url
    print('\n' + '='*65)
    print(f'  🛍️  Store widget  : {public_url}/')
    print(f'  🔧  Admin panel   : {public_url}/admin  (pass: kaggle-test-pass)')
    print(f'  📖  API docs      : {public_url}/docs')
    print('='*65)
else:
    print('ℹ️  No ngrok token — tests run against localhost.')
    print('   Paste your token in NGROK_TOKEN above for a shareable public URL.')

In [ ]:
# ── Step 8: Smoke test (EN / UR / AR) ────────────────────────────────────
import urllib.request, json

def chat(message):
    payload = json.dumps({'message': message}).encode()
    req = urllib.request.Request(
        'http://localhost:8000/api/chat',
        data=payload,
        headers={'Content-Type': 'application/json'}
    )
    with urllib.request.urlopen(req, timeout=120) as r:
        return json.loads(r.read())

tests = [
    'Hello!',
    'Do you have wireless headphones under 5000 PKR?',
    'What is your return policy?',
    'kya aap ke paas wireless headphones hain?',
    'هل لديكم سماعات بلوتوث لاسلكية؟',
]

for q in tests:
    try:
        resp = chat(q)
        products = [p['name'] for p in resp.get('products', [])]
        print(f'Q: {q[:60]}')
        print(f'A: {resp["answer"][:120]}')
        if products:
            print(f'   🛍️  {products}')
        print(f'   [{resp["lang"].upper()} | cached={resp["cached"]}]\n')
    except Exception as e:
        print(f'Q: {q}\n   ❌ {e}\n')

In [ ]:
# ── Step 9: Full eval harness (50 test cases) ────────────────────────────
import subprocess, json

result = subprocess.run(
    ['python', 'eval/run_eval.py',
     '--base-url', 'http://localhost:8000',
     '--out', '/tmp/eval_results.json'],
    cwd=str(REPO_ROOT),
    capture_output=True, text=True, timeout=900
)
print(result.stdout)
if result.returncode not in (0, 1) and result.stderr:
    print('STDERR:', result.stderr[-400:])

try:
    with open('/tmp/eval_results.json') as f:
        ev = json.load(f)
    print(f'Overall: {ev["correct"]}/{ev["total"]} ({ev["correct_pct"]}%)')
    print('By lang:', {k: f"{v["correct"]}/{v["total"]}" for k, v in ev["by_lang"].items()})
except:
    pass